In [8]:
import json
import sys

import pandas as pd

# notebooks run from notebooks/, so point Python one level up into ingestion/
sys.path.append("../ingestion")
from flatten_companyfacts import flatten

In [9]:
# Kraft Heinz: our known restatement case
with open("../data/raw/companyfacts/CIK0001637459.json") as f:
    raw = json.load(f)

df = flatten(raw)
df.shape

(22444, 14)

In [10]:
df.head(10)

,cik,entity_name,taxonomy,concept,unit,end,val,accn,fy,fp,form,filed,frame,start
0,1637459,Kraft Heinz Co,dei,EntityCommonStockSharesOutstanding,shares,2015-08-02,1.212833e+09,0001637459-15-000021,2015.0,Q2,10-Q,2015-08-10,CY2015Q2I,NaT
1,1637459,Kraft Heinz Co,dei,EntityCommonStockSharesOutstanding,shares,2015-11-01,1.213456e+09,0001637459-15-000049,2015.0,Q3,10-Q,2015-11-06,CY2015Q3I,NaT
2,1637459,Kraft Heinz Co,dei,EntityCommonStockSharesOutstanding,shares,2016-02-28,1.215190e+09,0001637459-16-000100,2015.0,FY,10-K,2016-03-03,NaN,NaT
3,1637459,Kraft Heinz Co,dei,EntityCommonStockSharesOutstanding,shares,2016-05-01,1.215955e+09,0001637459-16-000147,2016.0,Q1,10-Q,2016-05-05,CY2016Q1I,NaT
4,1637459,Kraft Heinz Co,dei,EntityCommonStockSharesOutstanding,shares,2016-07-31,1.217651e+09,0001637459-16-000179,2016.0,Q2,10-Q,2016-08-05,CY2016Q2I,NaT
5,1637459,Kraft Heinz Co,dei,EntityCommonStockSharesOutstanding,shares,2016-10-30,1.217270e+09,0001637459-16-000207,2016.0,Q3,10-Q,2016-11-04,CY2016Q3I,NaT
6,1637459,Kraft Heinz Co,dei,EntityCommonStockSharesOutstanding,shares,2017-02-18,1.217136e+09,0001637459-17-000007,2016.0,FY,10-K,2017-02-23,NaN,NaT
7,1637459,Kraft Heinz Co,dei,EntityCommonStockSharesOutstanding,shares,2017-04-29,1.217633e+09,0001637459-17-000081,2017.0,Q1,10-Q,2017-05-04,NaN,NaT
8,1637459,Kraft Heinz Co,dei,EntityCommonStockSharesOutstanding,shares,2017-04-29,1.217633e+09,0001637459-17-000116,2017.0,Q1,10-Q/A,2017-11-07,CY2017Q1I,NaT
9,1637459,Kraft Heinz Co,dei,EntityCommonStockSharesOutstanding,shares,2017-07-29,1.218251e+09,0001637459-17-000101,2017.0,Q2,10-Q,2017-08-04,NaN,NaT


## Find value changes across filings

Every 10-K/10-Q repeats prior periods as comparatives, so the same fact is reported several times.
A **fact** is identified by `(taxonomy, concept, unit, start, end)`. For each fact we order its reports
by `filed` and compare each value with the previous one. Any difference is a *candidate* revision;
deciding whether it is a real restatement or noise comes later.

- `dropna=False` in `groupby`: instant facts have no `start`, and pandas would otherwise drop them.
- `accn` is a tiebreaker in the sort, so two filings on the same day have a stable order.

In [11]:
FACT_KEY = ["taxonomy", "concept", "unit", "start", "end"]


def find_changes(df):
    """Return one row per value change between consecutive filings of the same fact."""
    df = df.sort_values(FACT_KEY + ["filed", "accn"]).copy()
    by_fact = df.groupby(FACT_KEY, dropna=False)

    # shift(1) puts the previous filing's value next to each row, within the same fact
    df["prev_val"] = by_fact["val"].shift(1)
    df["prev_accn"] = by_fact["accn"].shift(1)
    df["prev_filed"] = by_fact["filed"].shift(1)
    df["prev_form"] = by_fact["form"].shift(1)

    # first report of a fact has no predecessor (prev_val is NaN) and is not a change
    changes = df[df["prev_val"].notna() & (df["val"] != df["prev_val"])].copy()

    changes["abs_change"] = changes["val"] - changes["prev_val"]
    # pct is undefined when the old value was 0
    changes["pct_change"] = changes["abs_change"] / changes["prev_val"].abs().replace(0, float("nan"))

    return changes[FACT_KEY + ["prev_val", "val", "abs_change", "pct_change",
                               "prev_accn", "accn", "prev_form", "form", "prev_filed", "filed"]]


changes = find_changes(df)
changes.shape

(817, 15)

### Sanity check: the known restatement

Kraft Heinz restated FY2016 in its FY2018 10-K (filed June 2019). Net income should go from $3,632M to $3,596M.

In [12]:
khc_fy2016 = changes[(changes["concept"] == "NetIncomeLoss") & (changes["end"] == "2016-12-31")]
assert len(khc_fy2016) == 1, "known restatement not detected"
khc_fy2016

,taxonomy,concept,unit,start,end,prev_val,val,abs_change,pct_change,prev_accn,accn,prev_form,form,prev_filed,filed
12710,us-gaap,NetIncomeLoss,USD,2016-01-04,2016-12-31,3.632000e+09,3.596000e+09,-36000000.0,-0.009912,0001637459-18-000015,0001637459-19-000049,10-K,10-K,2018-02-16,2019-06-07


### How big are the changes?

Most changes are not restatements. The distribution of `pct_change` and the concepts that change most often
are the first clues for separating noise from real revisions.

In [13]:
print("changes:", len(changes))
print("\nabsolute % change, quantiles:")
print(changes["pct_change"].abs().quantile([0.1, 0.25, 0.5, 0.75, 0.9]).round(4))
print("\nconcepts that change most often:")
changes["concept"].value_counts().head(10)

changes: 817

absolute % change, quantiles:
0.10    0.0013
0.25    0.0053
0.50    0.0219
0.75    0.2000
0.90    1.4760
Name: pct_change, dtype: float64

concepts that change most often:


concept
OtherOperatingActivitiesCashFlowStatement               30
GrossProfit                                             21
OperatingIncomeLoss                                     21
SellingGeneralAndAdministrativeExpense                  21
OtherLiabilitiesCurrent                                 16
OtherNonoperatingIncomeExpense                          16
PaymentsForProceedsFromOtherInvestingActivities         16
ProceedsFromPaymentsForOtherFinancingActivities         16
IncreaseDecreaseInOtherCurrentAssets                    14
EffectiveIncomeTaxRateReconciliationOtherAdjustments    13
Name: count, dtype: int64

## Group changes into revision events

A restatement is one event (one filing) that changes many numbers at once. Grouping changes by the filing
that made them (`accn`) turns hundreds of rows into a few dozen events, and shows what each event looks like:
how many values it changed, across how many concepts, and by how much.

In [ ]:
def revision_events(changes):
    """One row per filing that changed previously reported values."""
    return (
        changes.groupby(["accn", "form", "filed"])
        .agg(
            n_changes=("val", "size"),
            n_concepts=("concept", "nunique"),
            median_abs_pct=("pct_change", lambda s: s.abs().median()),
        )
        .sort_values("n_changes", ascending=False)
        .reset_index()
    )


khc_events = revision_events(changes)
print("filings with changes:", len(khc_events))
khc_events.head(5)

**Kraft Heinz:** one filing stands out. The FY2018 10-K (filed 2019-06-07, the restatement) accounts for
519 of the 817 changes, across 119 concepts, with a median change of only ~1.2%.

That is the signature of an error restatement: **broad** (many line items at once) but **small** (each number
moves a little), because a correction ripples through cost of sales, gross profit, operating income, net income, etc.

## Apple: the noise case

Apple has no known error restatement, so its changes show what benign revisions look like.

In [ ]:
with open("../data/raw/companyfacts/CIK0000320193.json") as f:
    apple = flatten(json.load(f))

apple_changes = find_changes(apple)
apple_events = revision_events(apple_changes)
print("changes:", len(apple_changes), "| filings with changes:", len(apple_events))
apple_events.head(6)

Apple's biggest events are all benign, and they look different from Kraft Heinz's restatement:

| Filing | What happened | Signature |
|---|---|---|
| 10-K/A, 2010-01-25 | Retrospective adoption of new revenue-recognition rules (iPhone) | 62 changes, 32 concepts, large moves (median ~27%) |
| 10-K, 2020-10-30 | 4-for-1 stock split (Aug 2020) | Per-share / share values change by exactly 4× |
| 10-Q/10-K, 2014 | 7-for-1 stock split (Jun 2014) | Per-share / share values change by exactly 7× |

### Stock splits

A split multiplies share counts and divides per-share values by a whole number, so the ratio between
old and new value is (almost exactly) an integer ≥ 2, and only `shares` / `USD/shares` units are affected.

In [ ]:
def is_split_like(changes, tolerance=0.05):
    """True where a share-based value changed by a whole-number ratio (2x, 4x, 7x, ...)."""
    bigger = changes[["prev_val", "val"]].abs().max(axis=1)
    smaller = changes[["prev_val", "val"]].abs().min(axis=1)
    ratio = bigger / smaller.replace(0, float("nan"))

    share_based = changes["unit"].isin(["shares", "USD/shares"])
    whole_ratio = (ratio.round() >= 2) & ((ratio - ratio.round()).abs() < tolerance)
    return share_based & whole_ratio


apple_changes["split_like"] = is_split_like(apple_changes)
changes["split_like"] = is_split_like(changes)

print("Apple split-like changes:", apple_changes["split_like"].sum(), "of", len(apple_changes))
print("Kraft Heinz split-like changes:", changes["split_like"].sum(), "of", len(changes))
apple_changes[apple_changes["split_like"]].groupby("filed").size()

## Findings and proposed noise filters

**Split detection works:** 111 of Apple's 449 changes are split-like, all filed right after the 2014 (7:1)
and 2020 (4:1) splits. Only 1 of Kraft Heinz's 817 changes matches, so the filter barely touches the real restatement.

| Noise type | Example | Proposed rule for dbt |
|---|---|---|
| Stock split | Apple EPS ÷7 (2014), ÷4 (2020) | Share-based unit and old/new ratio is a whole number ≥ 2 → `benign: split` |
| Rounding | Apple FY2007 net income $3,495M → $3,496M | \|% change\| < 0.1% → `noise: rounding` (Apple 20 changes, Kraft Heinz 64) |
| Accounting-rule change | Apple 10-K/A, Jan 2010 | Not separable by numbers alone; needs filing text or labels |
| Reclassification | Kraft Heinz "Other…" line items | Changes within one filing that offset each other (net ≈ 0); to test |

**Event-level signals.** Judge a *filing*, not a single number: Kraft Heinz's restatement changed 519 values
across 119 concepts with a median move of ~1.2% (broad and small). Apple's benign events changed fewer values
by larger amounts.

**Ground truth.** An 8-K Item 4.02 (non-reliance) filing marks a Big R restatement. Kraft Heinz filed one on
2019-05-06, a month before the restated 10-K. These labels can validate the detector at scale.

**Open questions:** the Kraft Heinz 10-K filed 2018-02-16 (27 changes, median ~27%) is unexplained, and a
second Item 4.02 from 2017-11-06 hasn't been examined yet.